# AURA — Intent Classifier Training (XLM-RoBERTa)

Fine-tunes **`xlm-roberta-base`** to classify Roman Urdu / English / code-switched WhatsApp messages into 6 intents:
`book_appointment`, `cancel_booking`, `check_availability`, `price_query`, `greeting`, `reschedule`.

It then compares the trained model with **Gemini (zero-shot)** on the same test set. That comparison is the results table for the report.

### How to run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When asked, upload `train.json`, `val.json`, `test.json` (from `AURA/Data/processed/`)
4. When asked, allow Google Drive access (the trained model is saved there)
5. When asked, paste your Gemini API key (only needed for the comparison, section 7)

Total time: about 15–25 minutes.

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate scikit-learn google-genai

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

## 2. Upload the dataset

In [ ]:
import json, os
from google.colab import files

REQUIRED = ["train.json", "val.json", "test.json"]
missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing:
    print("Please upload:", missing)
    files.upload()

splits = {}
for name in ["train", "val", "test"]:
    with open(f"{name}.json", encoding="utf-8") as f:
        splits[name] = json.load(f)
    print(f"{name}: {len(splits[name])} messages")

## 3. Prepare labels and tokenize

In [ ]:
from collections import Counter
from transformers import AutoTokenizer, set_seed

SEED = 42
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 96
set_seed(SEED)

LABELS = sorted({row["intent"] for rows in splits.values() for row in rows})
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print("Labels:", LABELS)
for name, rows in splits.items():
    print(name, dict(Counter(r["intent"] for r in rows)))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.enc = tokenizer([r["text"] for r in rows], truncation=True, max_length=MAX_LEN)
        self.labels = [label2id[r["intent"]] for r in rows]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_ds, val_ds, test_ds = (IntentDataset(splits[s]) for s in ["train", "val", "test"])
longest = max(len(x) for x in train_ds.enc["input_ids"])
print(f"Longest message: {longest} tokens (limit {MAX_LEN})")

## 4. Train

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, EarlyStoppingCallback)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def compute_metrics(pred):
    y_pred = np.argmax(pred.predictions, axis=-1)
    return {"accuracy": accuracy_score(pred.label_ids, y_pred),
            "macro_f1": f1_score(pred.label_ids, y_pred, average="macro")}

import math
EPOCHS, BATCH = 12, 16
total_steps = math.ceil(len(train_ds) / BATCH) * EPOCHS
warmup_steps = int(0.1 * total_steps)   # 10% warm-up (works on transformers v4 and v5)

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()
print("Best validation:", trainer.evaluate())

## 5. Evaluate on the test set
The test set was never seen during training or model selection, so these are the numbers to report.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

out = trainer.predict(test_ds)
probs = torch.softmax(torch.tensor(out.predictions), dim=-1).numpy()
y_true = out.label_ids
y_pred = probs.argmax(-1)
confidence = probs.max(-1)

xlmr_acc = accuracy_score(y_true, y_pred)
xlmr_f1 = f1_score(y_true, y_pred, average="macro")
ALL_IDS = list(range(len(LABELS)))
report = classification_report(y_true, y_pred, labels=ALL_IDS, target_names=LABELS, digits=3, zero_division=0)
print(f"XLM-RoBERTa  accuracy={xlmr_acc:.3f}  macro-F1={xlmr_f1:.3f}\n")
print(report)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=ALL_IDS), display_labels=LABELS).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
ax.set_title("XLM-RoBERTa — test set confusion matrix")
plt.tight_layout(); plt.savefig("confusion_matrix_xlmr.png", dpi=150); plt.show()

# Mistakes, for error analysis in the report
errors = [{"text": splits["test"][i]["text"], "true": id2label[int(y_true[i])],
           "predicted": id2label[int(y_pred[i])], "confidence": round(float(confidence[i]), 3)}
          for i in range(len(y_true)) if y_true[i] != y_pred[i]]
print(f"\n{len(errors)} mistakes:")
for e in errors:
    print(f"  [{e['true']} -> {e['predicted']} @ {e['confidence']}] {e['text']}")

### Choosing the confidence threshold
AURA hands a message to Gemini (and eventually the owner) when the model is unsure.
This table shows, for each threshold, how many messages the model would answer by itself and how accurate those answers are.

In [ ]:
print(f"{'threshold':>9} | {'answered by model':>17} | {'accuracy on those':>17}")
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = confidence >= t
    acc = accuracy_score(y_true[keep], y_pred[keep]) if keep.any() else float("nan")
    print(f"{t:>9} | {keep.mean():>16.1%} | {acc:>17.3f}")

## 6. Save the model to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/AURA/intent_model"
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)          # weights + config (with label names)
tokenizer.save_pretrained(SAVE_DIR)
!cp confusion_matrix_xlmr.png "$SAVE_DIR/"
with open(f"{SAVE_DIR}/test_errors.json", "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)
with open(f"{SAVE_DIR}/classification_report.txt", "w") as f:
    f.write(report)
print("Saved to", SAVE_DIR)
!ls -lh "$SAVE_DIR"

### Speed on CPU
The server AURA runs on has no GPU, so this measures the real response time.

In [ ]:
import time
cpu_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).eval()
cpu_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
sample = [r["text"] for r in splits["test"][:50]]
with torch.no_grad():
    cpu_model(**cpu_tok(sample[0], return_tensors="pt"))   # warm-up
    t0 = time.perf_counter()
    for s in sample:
        cpu_model(**cpu_tok(s, return_tensors="pt", truncation=True, max_length=MAX_LEN))
xlmr_ms = (time.perf_counter() - t0) / len(sample) * 1000
print(f"XLM-RoBERTa on CPU: {xlmr_ms:.0f} ms per message")

## 7. Comparison with Gemini (zero-shot)
Sends each test message to Gemini with only the list of intent names, the same way the original AURA prototype worked.
This takes roughly 10 minutes because of free-tier rate limits. You can skip it and run it later.

In [ ]:
from getpass import getpass
from google import genai

GEMINI_MODEL = "gemini-3.5-flash-lite"   # same model AURA uses; change if needed
client = genai.Client(api_key=getpass("Paste your Gemini API key: "))
DELAY = 4.5   # seconds between calls, keeps under the free-tier rate limit

PROMPT = '''Classify this WhatsApp message sent to a small business in Pakistan.
It may be Roman Urdu, English, or mixed.
Reply with exactly one label from: {labels}
Message: "{text}"
Label:'''

def gemini_intent(text, retries=4):
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            r = client.models.generate_content(model=GEMINI_MODEL,
                    contents=PROMPT.format(labels=", ".join(LABELS), text=text))
            ms = (time.perf_counter() - t0) * 1000
            answer = (r.text or "").strip().lower()
            label = next((l for l in LABELS if l in answer), "unknown")
            return label, ms
        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"  retry in {wait}s ({str(e)[:80]})")
            time.sleep(wait)
    return "unknown", float("nan")

gem_pred, gem_ms = [], []
for i, row in enumerate(splits["test"]):
    label, ms = gemini_intent(row["text"])
    gem_pred.append(label); gem_ms.append(ms)
    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{len(splits['test'])} done")
    time.sleep(DELAY)

gem_true = [r["intent"] for r in splits["test"]]
gem_acc = accuracy_score(gem_true, gem_pred)
gem_f1 = f1_score(gem_true, gem_pred, labels=LABELS, average="macro")
print(f"\nGemini  accuracy={gem_acc:.3f}  macro-F1={gem_f1:.3f}  (unparseable answers: {gem_pred.count('unknown')})")
print(classification_report(gem_true, gem_pred, labels=LABELS, digits=3, zero_division=0))

## 8. Results table for the report

In [ ]:
import pandas as pd
rows = [{"Model": "XLM-RoBERTa (fine-tuned)", "Accuracy": xlmr_acc, "Macro F1": xlmr_f1,
         "Latency (ms/msg)": xlmr_ms, "Cost": "Free (runs locally)"}]
if "gem_acc" in globals():
    rows.append({"Model": f"Gemini zero-shot ({GEMINI_MODEL})", "Accuracy": gem_acc, "Macro F1": gem_f1,
                 "Latency (ms/msg)": float(np.nanmean(gem_ms)), "Cost": "API call per message"})
else:
    print("Gemini comparison (section 7) was skipped - showing XLM-RoBERTa only.")
results = pd.DataFrame(rows).round(3)
display(results)
results.to_csv(f"{SAVE_DIR}/comparison.csv", index=False)
with open(f"{SAVE_DIR}/metrics.json", "w") as f:
    json.dump(results.to_dict(orient="records"), f, indent=2)
print("Saved comparison.csv and metrics.json to", SAVE_DIR)

## 9. Try your own messages

In [ ]:
def predict(text):
    with torch.no_grad():
        p = torch.softmax(cpu_model(**cpu_tok(text, return_tensors="pt")).logits, -1)[0]
    i = int(p.argmax())
    return id2label[i], round(float(p[i]), 3)

for msg in ["kal 3 baje haircut book kar do",
            "meri booking cancel kar dein please",
            "checkup ki fees kitni hai?",
            "kya aaj sham ko koi slot khali hai?",
            "appointment Friday se Monday shift kar dein",
            "Assalam o alaikum"]:
    print(predict(msg), "<-", msg)